# Notebook 05 — KPI Computation & Final Load Prep
**Purpose:** Compute all KPIs, export Tableau-ready CSVs, verify data quality before dashboard build.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
data_path = '../data/processed/cleaned_data.csv'
if not os.path.exists(data_path):
    data_path = '../Project/cleaned_data.csv'

df = pd.read_csv(data_path, encoding='latin1')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID']  = df['CustomerID'].astype(int)
df['Revenue']     = df['Quantity'] * df['UnitPrice']
df['YearMonth']   = df['InvoiceDate'].dt.to_period('M').astype(str)
df['Quarter']     = df['InvoiceDate'].dt.quarter
df['Year']        = df['InvoiceDate'].dt.year
df['Hour']        = df['InvoiceDate'].dt.hour
df['DayOfWeek']   = df['InvoiceDate'].dt.day_name()

# Exclude partial Dec 2011
df_full = df[~((df['Year'] == 2011) & (df['InvoiceDate'].dt.month == 12))].copy()

print(f'Rows (full months): {len(df_full):,}')
print(f'Date range: {df_full["InvoiceDate"].min().date()} → {df_full["InvoiceDate"].max().date()}')

## 2. Core KPI Computation

In [ ]:
# --- KPI 1: Total Revenue ---
total_revenue = df_full['Revenue'].sum()

# --- KPI 2: Total Orders (unique invoices) ---
total_orders = df_full['InvoiceNo'].nunique()

# --- KPI 3: Average Order Value (AOV) ---
aov = df_full.groupby('InvoiceNo')['Revenue'].sum()
avg_order_value = aov.mean()

# --- KPI 4: Total Unique Customers ---
total_customers = df_full['CustomerID'].nunique()

# --- KPI 5: Repeat Purchase Rate ---
purchase_counts = df_full.groupby('CustomerID')['InvoiceNo'].nunique()
repeat_customers = (purchase_counts > 1).sum()
repeat_rate = repeat_customers / total_customers * 100

# --- KPI 6: Average Revenue per Customer ---
avg_rev_per_customer = df_full.groupby('CustomerID')['Revenue'].sum().mean()

# --- KPI 7: Revenue per Month (avg) ---
monthly_rev = df_full.groupby('YearMonth')['Revenue'].sum()
avg_monthly_revenue = monthly_rev.mean()

# --- KPI 8: Monthly Revenue Growth % (avg MoM) ---
mom_growth = monthly_rev.sort_index().pct_change().mean() * 100

# --- KPI 9: Top Country Revenue Share ---
uk_share = df_full[df_full['Country']=='United Kingdom']['Revenue'].sum() / total_revenue * 100

# --- KPI 10: Units Sold ---
total_units = df_full['Quantity'].sum()

kpis = {
    'Total Revenue (£)':            f'{total_revenue:,.2f}',
    'Total Orders':                 f'{total_orders:,}',
    'Average Order Value (£)':      f'{avg_order_value:.2f}',
    'Total Unique Customers':       f'{total_customers:,}',
    'Repeat Purchase Rate (%)':     f'{repeat_rate:.1f}',
    'Avg Revenue per Customer (£)': f'{avg_rev_per_customer:.2f}',
    'Avg Monthly Revenue (£)':      f'{avg_monthly_revenue:,.2f}',
    'Avg MoM Revenue Growth (%)':   f'{mom_growth:.1f}',
    'UK Revenue Share (%)':         f'{uk_share:.1f}',
    'Total Units Sold':             f'{total_units:,}',
}

print('=== KPI DASHBOARD ===')
for k, v in kpis.items():
    print(f'  {k}: {v}')

## 3. Export Tableau-Ready Datasets

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

# --- Export 1: Monthly Revenue with Growth ---
monthly_df = df_full.groupby('YearMonth').agg(
    Revenue       = ('Revenue', 'sum'),
    Orders        = ('InvoiceNo', 'nunique'),
    Customers     = ('CustomerID', 'nunique'),
    Units_Sold    = ('Quantity', 'sum')
).reset_index().sort_values('YearMonth')
monthly_df['MoM_Growth_Pct'] = monthly_df['Revenue'].pct_change() * 100
monthly_df['AOV'] = monthly_df['Revenue'] / monthly_df['Orders']
monthly_df.to_csv('../data/processed/kpi_monthly.csv', index=False)
print(f'kpi_monthly.csv: {len(monthly_df)} rows')

# --- Export 2: Revenue by Country ---
country_df = df_full.groupby('Country').agg(
    Revenue   = ('Revenue', 'sum'),
    Orders    = ('InvoiceNo', 'nunique'),
    Customers = ('CustomerID', 'nunique')
).reset_index().sort_values('Revenue', ascending=False)
country_df['Revenue_Share_Pct'] = country_df['Revenue'] / country_df['Revenue'].sum() * 100
country_df.to_csv('../data/processed/kpi_country.csv', index=False)
print(f'kpi_country.csv: {len(country_df)} rows')

# --- Export 3: Top Products ---
product_df = df_full.groupby(['StockCode', 'Description']).agg(
    Revenue       = ('Revenue', 'sum'),
    Units_Sold    = ('Quantity', 'sum'),
    Num_Invoices  = ('InvoiceNo', 'nunique')
).reset_index().sort_values('Revenue', ascending=False).head(100)
product_df.to_csv('../data/processed/kpi_top_products.csv', index=False)
print(f'kpi_top_products.csv: top 100 products')

# --- Export 4: Customer Summary ---
customer_df = df_full.groupby('CustomerID').agg(
    Total_Revenue  = ('Revenue', 'sum'),
    Num_Orders     = ('InvoiceNo', 'nunique'),
    Num_Items      = ('Quantity', 'sum'),
    First_Purchase = ('InvoiceDate', 'min'),
    Last_Purchase  = ('InvoiceDate', 'max'),
    Country        = ('Country', 'first')
).reset_index()
customer_df['AOV'] = customer_df['Total_Revenue'] / customer_df['Num_Orders']
customer_df['Tenure_Days'] = (customer_df['Last_Purchase'] - customer_df['First_Purchase']).dt.days
customer_df.to_csv('../data/processed/kpi_customers.csv', index=False)
print(f'kpi_customers.csv: {len(customer_df)} customers')

# --- Export 5: Day of Week Revenue ---
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_df = df_full.groupby('DayOfWeek').agg(
    Revenue = ('Revenue', 'sum'),
    Orders  = ('InvoiceNo', 'nunique')
).reset_index()
dow_df['DayOfWeek'] = pd.Categorical(dow_df['DayOfWeek'], categories=dow_order, ordered=True)
dow_df = dow_df.sort_values('DayOfWeek')
dow_df.to_csv('../data/processed/kpi_day_of_week.csv', index=False)
print(f'kpi_day_of_week.csv: {len(dow_df)} rows')

print('\nAll Tableau-ready exports complete → data/processed/')

## 4. Data Quality Verification

In [ ]:
print('=== FINAL DATA QUALITY CHECK ===')
print(f'Main dataset rows:    {len(df_full):,}')
print(f'Null values:')
print(df_full[['InvoiceNo','CustomerID','Revenue','Country']].isnull().sum().to_string())
print(f'\nNegative Revenue rows: {(df_full["Revenue"] <= 0).sum()}')
print(f'Negative Quantity rows: {(df_full["Quantity"] <= 0).sum()}')
print(f'\nExported files:')
for f in ['kpi_monthly.csv','kpi_country.csv','kpi_top_products.csv','kpi_customers.csv','kpi_day_of_week.csv']:
    path = f'../data/processed/{f}'
    if os.path.exists(path):
        tmp = pd.read_csv(path)
        print(f'  {f}: {len(tmp)} rows × {len(tmp.columns)} cols — OK')
    else:
        print(f'  {f}: MISSING')

print('\nPipeline complete. Ready for Tableau.')